# AIA Supervisor Agent — Test Notebook

This notebook contains all test and demo cells for the **AIA Supervisor Agent**.
It loads the full agent implementation from `agent_code.py` via `%run`, so all
functions, state, and the compiled LangGraph are available here without duplication.

**Sections:**
1. Setup — load agent code
2. Helper: `inspect_memory_tables`
3. Memory Lifecycle Demo (short-term · long-term · episodic)
4. Multi-Domain Genie Space Routing (Claims · Policies · Distribution · Customers)
5. Multi-Turn Conversation Test


## 1. Setup — Load Agent Code

Runs `agent_code.py` so every function, the compiled LangGraph, and `agent` are
available in this session without re-importing anything manually.


In [ ]:
%run ./agent_code

In [ ]:
# Verify the agent is loaded
print(f'Agent: {type(agent).__name__}')
print(f'Graph nodes: {list(graph.nodes)}')
print(f'LLM endpoint: {MODEL_ENDPOINT}')


## 2. Helper: `inspect_memory_tables`

Queries all three memory tables for a given `thread_id` / `user_id` and prints a
structured summary. Reused across every test section below.


In [ ]:
def inspect_memory_tables(thread_id, user_id, label=''):
    """Query and display all three memory tables for a thread / user."""
    sep = '=' * 70
    print(f'\n{sep}')
    print(f"  MEMORY STATE{f' — {label}' if label else ''}")
    print(sep)

    # 1. Short-term memory
    print('\n[1] SHORT-TERM MEMORY (ai_ops.conversations)')
    try:
        conv = _run_sql(
            f"SELECT thread_id, checkpoint_id, created_at, LEFT(state_json, 200) AS state_preview "
            f"FROM {CATALOG}.ai_ops.conversations "
            f"WHERE thread_id = '{thread_id}' ORDER BY created_at DESC LIMIT 5"
        )
        if conv['rows']:
            for r in conv['rows']:
                print(f"  checkpoint={r['checkpoint_id']}  created={r['created_at']}")
                print(f"    preview: {r['state_preview']}...")
            print(f"  -> {len(conv['rows'])} checkpoint(s) found")
        else:
            print('  -> No checkpoints found (empty)')
    except Exception as e:
        print(f'  -> Table not available: {str(e)[:100]}')

    # 2. Long-term memory
    print('\n[2] LONG-TERM MEMORY (ai_ops.user_memory)')
    try:
        mem = _run_sql(
            f"SELECT memory_key, memory_value, memory_type, confidence, updated_at "
            f"FROM {CATALOG}.ai_ops.user_memory "
            f"WHERE user_id = '{user_id}' ORDER BY updated_at DESC"
        )
        if mem['rows']:
            for r in mem['rows']:
                print(f"  {r['memory_key']:25s} = {r['memory_value']:30s}  "
                      f"(type={r['memory_type']}, conf={r['confidence']})")
            print(f"  -> {len(mem['rows'])} memory entries found")
        else:
            print('  -> No user memories found (empty)')
    except Exception as e:
        print(f'  -> Table not available: {str(e)[:100]}')

    # 3. Episodic memory
    print('\n[3] EPISODIC MEMORY (ai_ops.episodic_memory)')
    try:
        ep = _run_sql(
            f"SELECT episode_id, question, intent, domain, agents_used, outcome, lesson_learned, created_at "
            f"FROM {CATALOG}.ai_ops.episodic_memory "
            f"WHERE thread_id = '{thread_id}' ORDER BY created_at DESC LIMIT 5"
        )
        if ep['rows']:
            for r in ep['rows']:
                print(f"  episode={r['episode_id']}  intent={r['intent']}  "
                      f"domain={r['domain']}  outcome={r['outcome']}")
                print(f"    Q: {r['question'][:80]}")
                print(f"    agents: {r['agents_used']}  lesson: {r.get('lesson_learned', 'None')}")
            print(f"  -> {len(ep['rows'])} episode(s) found")
        else:
            print('  -> No episodes found (empty)')
    except Exception as e:
        print(f'  -> Table not available: {str(e)[:100]}')

    print(f'\n{sep}\n')

print('inspect_memory_tables() defined')


---

## 3. Memory Lifecycle Demo

Demonstrates **end-to-end memory population and retrieval** across the three layers:

| Layer | Table | Purpose |
|-------|-------|---------|
| **Short-term** | `ai_ops.conversations` | Delta checkpoints for multi-turn context |
| **Long-term** | `ai_ops.user_memory` | Persistent user preferences & facts |
| **Episodic** | `ai_ops.episodic_memory` | Interaction logs & lessons learned |

**Flow:**
1. Clean prior demo data → check baseline
2. Conversational request with personal facts → writes to all 3 tables
3. Inspect tables
4. Follow-up document lookup → reads short-term + long-term, writes new checkpoint & episode
5. Inspect tables
6. Third KPI request → all 3 layers read + written
7. Final inspection & summary


### Step 1 — Prepare: Clean Prior Demo Data & Check Baseline


In [ ]:
DEMO_THREAD_ID = 'memory-demo-thread-001'
DEMO_USER_ID = 'demo-user-sarah'

print('Cleaning up prior demo data...')
_cleanup = [
    (f"DELETE FROM {CATALOG}.ai_ops.conversations WHERE thread_id = '{DEMO_THREAD_ID}'", 'conversations'),
    (f"DELETE FROM {CATALOG}.ai_ops.user_memory WHERE user_id = '{DEMO_USER_ID}'", 'user_memory'),
    (f"DELETE FROM {CATALOG}.ai_ops.episodic_memory WHERE thread_id = '{DEMO_THREAD_ID}'", 'episodic_memory'),
]
for _sql, _label in _cleanup:
    try:
        _run_sql(_sql)
        print(f'  Cleared {_label}')
    except Exception as _e:
        print(f'  Warning — could not clear {_label}: {_e}')

inspect_memory_tables(DEMO_THREAD_ID, DEMO_USER_ID, label='BASELINE (before any requests)')


### Step 2 — First Request: Conversational Message with Personal Facts

Intent: `conversational` → agent **writes** a checkpoint, **extracts & saves** user facts
(name, role, region, product preference), and **logs** an episode.


In [ ]:
request_1 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        "Hi, I'm Sarah, a regional claims manager based in Singapore. "
        'I prefer concise responses and usually focus on the Health and '
        'Critical Illness product lines.'
    )}],
    custom_inputs={'thread_id': DEMO_THREAD_ID, 'user_id': DEMO_USER_ID}
)

print('Sending Request 1 (conversational with personal facts)...')
print(f'  thread_id: {DEMO_THREAD_ID}  |  user_id: {DEMO_USER_ID}')
print(f"  message: {getattr(request_1.input[0], 'content', '')}\n")

response_1 = agent.predict(request_1)

for item in response_1.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print('\n=== METADATA ===')
            print(f"  Intent:     {meta.get('intent')}")
            print(f"  Nodes:      {meta.get('nodes_executed')}")
            print(f"  Checkpoint: {meta.get('checkpoint_id')}")
        except json.JSONDecodeError:
            print(f'\n=== METADATA (raw) ===\n  {text[:300]}')


### Step 3 — Inspect Memory Tables After Request 1

Expected: **1 checkpoint**, extracted user facts (name, role, region, product), **1 episode** (`intent=conversational`).


In [ ]:
import time
time.sleep(3)  # allow async writes to settle
inspect_memory_tables(DEMO_THREAD_ID, DEMO_USER_ID, label='AFTER REQUEST 1 (conversational intro)')


### Step 4 — Second Request: Document Lookup on the Same Thread

Uses **short-term memory** (checkpoint) for conversation context + **long-term memory**
(Sarah's preferences) for personalisation. Writes a new checkpoint and a new episode.


In [ ]:
request_2 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': 'What are the exclusions for the Critical Illness plans?'}],
    custom_inputs={'thread_id': DEMO_THREAD_ID, 'user_id': DEMO_USER_ID}
)

print('Sending Request 2 (document lookup — same thread)...')
print(f'  thread_id: {DEMO_THREAD_ID}  (loads prior checkpoint)')
print(f'  user_id:   {DEMO_USER_ID}    (loads saved preferences)')
print(f"  message: {getattr(request_2.input[0], 'content', '')}\n")

_memory_cache_ts = 0  # force fresh load from tables

response_2 = agent.predict(request_2)

for item in response_2.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print('\n=== METADATA ===')
            print(f"  Intent:     {meta.get('intent')}")
            print(f"  Nodes:      {meta.get('nodes_executed')}")
            print(f"  Checkpoint: {meta.get('checkpoint_id')}")
        except json.JSONDecodeError:
            print(f'\n=== METADATA (raw) ===\n  {text[:300]}')


### Step 5 — Inspect Memory Tables After Request 2

Expected: **2 checkpoints** (latest has 2-turn history), user memory unchanged,
**2 episodes** (second has `intent=document_lookup`).


In [ ]:
time.sleep(3)
inspect_memory_tables(DEMO_THREAD_ID, DEMO_USER_ID, label='AFTER REQUEST 2 (document lookup)')

print('--- Verifying Short-Term Memory Contains Prior Conversation ---')
checkpoint = _load_checkpoint(DEMO_THREAD_ID)
if checkpoint and checkpoint.get('messages'):
    print(f"  Checkpoint has {len(checkpoint['messages'])} messages:")
    for i, msg in enumerate(checkpoint['messages']):
        print(f"    [{i}] {msg.get('role', '?')}: {msg.get('content', '')[:100]}...")
    print(f"  Stored intent: {checkpoint.get('intent')}")
    print(f"  Stored domain: {checkpoint.get('domain')}")
else:
    print('  No checkpoint found')


### Step 6 — Third Request: All Memory Layers Working Together

**Short-term** loads 4-message history · **Long-term** personalises with Sarah's prefs ·
**Episodic** injects prior lessons from the `document_lookup` episode.


In [ ]:
request_3 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'How do I file a claim under the Critical Illness plans? '
        'Also, what is the waiting period?'
    )}],
    custom_inputs={'thread_id': DEMO_THREAD_ID, 'user_id': DEMO_USER_ID}
)

print('Sending Request 3 (follow-up using all memory layers)...')
print(f'  thread_id: {DEMO_THREAD_ID}')
print(f'  user_id:   {DEMO_USER_ID}')
print(f"  message: {getattr(request_3.input[0], 'content', '')}\n")

_memory_cache_ts = 0

response_3 = agent.predict(request_3)

for item in response_3.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
        print('\n--- Personalization Checks ---')
        lower = text.lower()
        checks = {
            "Addressed by name ('Sarah')": 'sarah' in lower,
            'Concise reply (< 500 chars)': len(text) < 500,
            'Mentions Critical Illness': 'critical illness' in lower,
        }
        for check, passed in checks.items():
            print(f"  [{'PASS' if passed else 'CHECK'}] {check}")
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print('\n=== METADATA ===')
            print(f"  Intent:     {meta.get('intent')}")
            print(f"  Nodes:      {meta.get('nodes_executed')}")
            print(f"  Checkpoint: {meta.get('checkpoint_id')}")
        except json.JSONDecodeError:
            print(f'\n=== METADATA (raw) ===\n  {text[:300]}')


### Step 7 — Final Inspection & Summary

| Table | Expected |
|-------|----------|
| `ai_ops.conversations` | 3 checkpoints |
| `ai_ops.user_memory` | Facts extracted from Request 1 (name, role, region, product pref) |
| `ai_ops.episodic_memory` | 3 episodes: `conversational` → `document_lookup` → `document_lookup` |


In [ ]:
time.sleep(3)
inspect_memory_tables(DEMO_THREAD_ID, DEMO_USER_ID, label='FINAL STATE (after 3 requests)')

print('\n' + '=' * 70)
print('  MEMORY LIFECYCLE SUMMARY')
print('=' * 70)

try:
    c = _run_sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.ai_ops.conversations WHERE thread_id = '{DEMO_THREAD_ID}'")['rows'][0]['cnt']
    m = _run_sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.ai_ops.user_memory WHERE user_id = '{DEMO_USER_ID}'")['rows'][0]['cnt']
    e = _run_sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.ai_ops.episodic_memory WHERE thread_id = '{DEMO_THREAD_ID}'")['rows'][0]['cnt']
    print(f"""
  Requests sent:                3
  ─────────────────────────────────────────────
  Conversation checkpoints:     {c}  (short-term memory)
  User memory entries:          {m}  (long-term memory)
  Episodic memory episodes:     {e}  (episodic memory)
  ─────────────────────────────────────────────

  Request 1 — conversational intro:
    WRITE: checkpoint, user facts, episode
    READ:  none (first interaction)

  Request 2 — document lookup:
    WRITE: new checkpoint, new episode
    READ:  checkpoint (prior context), user_memory (preferences)

  Request 3 — follow-up CI query:
    WRITE: new checkpoint, new episode
    READ:  checkpoint (4-message history), user_memory (name + prefs), episodic lessons
""")
except Exception as e:
    print(f'  Could not generate summary: {str(e)[:200]}')


### Cleanup (Optional) — Reset Demo Data

Uncomment and run to wipe all demo data so the test can be re-run from scratch.


In [ ]:
# _run_sql(f"DELETE FROM {CATALOG}.ai_ops.conversations WHERE thread_id = '{DEMO_THREAD_ID}'")
# _run_sql(f"DELETE FROM {CATALOG}.ai_ops.user_memory WHERE user_id = '{DEMO_USER_ID}'")
# _run_sql(f"DELETE FROM {CATALOG}.ai_ops.episodic_memory WHERE thread_id = '{DEMO_THREAD_ID}'")
# _memory_cache_ts = 0
# print('Demo data cleaned up. Ready to re-run.')


---

## 4. Multi-Domain Genie Space Routing

Each query targets a different Genie Space registered in the Context Index.
The supervisor classifies intent as `simple_kpi`, resolves the best-matching
space via Vector Search, and routes to `route_to_genie`.

| # | Target Space | Domain |
|---|---|---|
| 1 | Claims Analytics | `claims` |
| 2 | Policy & Underwriting | `policies` |
| 3 | Distribution & Channels | `distribution` |
| 4 | Customer Analytics | `customers` |


In [ ]:
GENIE_THREAD_ID = 'genie-routing-demo-thread'
GENIE_USER_ID = 'demo-user-genie'

def _print_routing_response(response, checks: dict):
    """Print agent response + routing checks from a ResponsesAgentResponse."""
    for item in response.output:
        item_id = getattr(item, 'id', '')
        text = getattr(item, 'text', '')
        if item_id == 'msg_answer':
            print('=== AGENT RESPONSE ===')
            print(text)
            print('\n--- Routing Checks ---')
            lower = text.lower()
            for label, condition in checks.items():
                passed = condition(lower, text)
                print(f"  [{'PASS' if passed else 'CHECK'}] {label}")
        elif item_id == 'msg_metadata' and text.strip():
            try:
                meta = json.loads(text)
                genie = meta.get('agent_details', {}).get('genie', {})
                print('\n=== METADATA ===')
                print(f"  Intent: {meta.get('intent')}  |  Domain: {meta.get('domain')}")
                print(f"  Nodes:  {meta.get('nodes_executed')}")
                print(f"  Genie status:     {genie.get('status', 'N/A')}")
                print(f"  Genie space used: {genie.get('space_id', 'N/A')}")
                print(f"  Genie display:    {genie.get('display_name', 'N/A')}")
                print(f"  SQL preview:      {str(genie.get('sql', 'N/A'))[:120]}")
            except json.JSONDecodeError:
                print(f'\n=== METADATA (raw) ===\n  {text[:300]}')

print('Routing helper _print_routing_response() defined')


### Query 1 — Claims Analytics Space

Target: Claims Analytics (`01f1272d4ba6144ba75d868762f1925d`).
Semantic match: *claims*, *region*, *count*.


In [ ]:
_memory_cache_ts = 0

request_claims = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'What is the total number of claims by region for the last 12 months?'
    )}],
    custom_inputs={'thread_id': GENIE_THREAD_ID, 'user_id': GENIE_USER_ID}
)

print('Sending Claims Analytics query...')
print(f"  message: {getattr(request_claims.input[0], 'content', '')}\n")

response_claims = agent.predict(request_claims)

_print_routing_response(response_claims, checks={
    'Mentions claims': lambda lower, _: 'claim' in lower,
    'Mentions region': lambda lower, _: 'region' in lower,
    'Contains numeric data': lambda _, text: any(c.isdigit() for c in text),
})


### Query 2 — Policy & Underwriting Space

Target: Policy & Underwriting (`01f1272d4c6b1fb49223785ab841befd`).
Semantic match: *premium*, *distribution channel*.


In [ ]:
_memory_cache_ts = 0

request_policies = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'What is the total premium volume by distribution channel?'
    )}],
    custom_inputs={'thread_id': GENIE_THREAD_ID, 'user_id': GENIE_USER_ID}
)

print('Sending Policy & Underwriting query...')
print(f"  message: {getattr(request_policies.input[0], 'content', '')}\n")

response_policies = agent.predict(request_policies)

_print_routing_response(response_policies, checks={
    'Mentions premium': lambda lower, _: 'premium' in lower,
    'Mentions policy/policies': lambda lower, _: 'policy' in lower or 'policies' in lower,
    'Contains numeric data': lambda _, text: any(c.isdigit() for c in text),
})


### Query 3 — Distribution & Channels Space

Target: Distribution & Channels (`01f1272d4d271203ad122e9280470248`).
Semantic match: *top-performing agents*, *channel contribution*, *commission*.


In [ ]:
_memory_cache_ts = 0

request_distribution = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'Who are the top 10 performing agents by premium collected? '
        'Also show me channel contribution percentages and commission breakdown for last quarter.'
    )}],
    custom_inputs={'thread_id': GENIE_THREAD_ID, 'user_id': GENIE_USER_ID}
)

print('Sending Distribution & Channels query...')
print(f"  message: {getattr(request_distribution.input[0], 'content', '')}\n")

response_distribution = agent.predict(request_distribution)

_print_routing_response(response_distribution, checks={
    'Mentions agent(s)': lambda lower, _: 'agent' in lower,
    'Mentions channel': lambda lower, _: 'channel' in lower,
    'Mentions premium or commission': lambda lower, _: 'premium' in lower or 'commission' in lower,
})


### Query 4 — Customer Analytics Space

Target: Customer Analytics (`01f1272d4de1188cac8feeb7e71bdb69`).
Semantic match: *customer segments*, *retention rate*, *demographics*.


In [ ]:
_memory_cache_ts = 0

request_customers = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'Which customer segments have the highest claim frequency? '
        'What is the retention rate by segment and show the demographic breakdown '
        'of our top-tier customers.'
    )}],
    custom_inputs={'thread_id': GENIE_THREAD_ID, 'user_id': GENIE_USER_ID}
)

print('Sending Customer Analytics query...')
print(f"  message: {getattr(request_customers.input[0], 'content', '')}\n")

response_customers = agent.predict(request_customers)

_print_routing_response(response_customers, checks={
    'Mentions customer': lambda lower, _: 'customer' in lower,
    'Mentions segment': lambda lower, _: 'segment' in lower,
    'Mentions retention': lambda lower, _: 'retention' in lower or 'retain' in lower,
})


### Post-Routing Memory Inspection

Inspect memory state after the four Genie routing queries.


In [ ]:
time.sleep(3)
inspect_memory_tables(GENIE_THREAD_ID, GENIE_USER_ID, label='AFTER GENIE ROUTING QUERIES')

print('--- Latest checkpoint ---')
checkpoint = _load_checkpoint(GENIE_THREAD_ID)
if checkpoint and checkpoint.get('messages'):
    print(f"  {len(checkpoint['messages'])} messages in latest checkpoint")
    for i, msg in enumerate(checkpoint['messages']):
        print(f"    [{i}] {msg.get('role', '?')}: {msg.get('content', '')[:100]}...")
    print(f"  intent={checkpoint.get('intent')}  domain={checkpoint.get('domain')}")
else:
    print('  No checkpoint found')


---

## 5. Multi-Turn Conversation Test

Three back-to-back queries on the **same thread** to verify:
- Context carry-over (Turn 3 cross-references results from Turns 1 & 2)
- Short-term memory checkpoint growth
- Correct intent resolution of follow-up shorthand questions


In [ ]:
CONV_THREAD_ID = 'conv-test-thread-001'
CONV_USER_ID = 'conv-test-user-001'
_memory_cache_ts = 0


### Turn 1 — Claims by Region (last 3 months)


In [ ]:
request_turn1 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'What is the total number of claims submitted by region for the last 3 months?'
    )}],
    custom_inputs={'thread_id': CONV_THREAD_ID, 'user_id': CONV_USER_ID}
)

print('[Turn 1] Claims by region...')
response_turn1 = agent.predict(request_turn1)

for item in response_turn1.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print(f"  Intent: {meta.get('intent')}  Domain: {meta.get('domain')}")
            print(f"  Nodes:  {meta.get('nodes_executed')}")
        except json.JSONDecodeError:
            pass


### Turn 2 — Premium by Product Type (same thread)


In [ ]:
_memory_cache_ts = 0

request_turn2 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'Show me the total premium collected by product type across all regions.'
    )}],
    custom_inputs={'thread_id': CONV_THREAD_ID, 'user_id': CONV_USER_ID}
)

print('[Turn 2] Premium by product type...')
response_turn2 = agent.predict(request_turn2)

for item in response_turn2.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print(f"  Intent: {meta.get('intent')}  Domain: {meta.get('domain')}")
            print(f"  Nodes:  {meta.get('nodes_executed')}")
        except json.JSONDecodeError:
            pass


### Turn 3 — Cross-Reference Follow-Up (tests context carry-over)

This follow-up question references both previous results. The agent must load the
short-term checkpoint to resolve the context correctly.


In [ ]:
_memory_cache_ts = 0

request_turn3 = ResponsesAgentRequest(
    input=[{'role': 'user', 'content': (
        'Based on those two results — the claims by region and premium by product — '
        'which regions are generating the most premium but also have the highest claim volumes? '
        'Are there any regions where we might be underpriced?'
    )}],
    custom_inputs={'thread_id': CONV_THREAD_ID, 'user_id': CONV_USER_ID}
)

print('[Turn 3] Cross-reference follow-up...')
response_turn3 = agent.predict(request_turn3)

for item in response_turn3.output:
    item_id = getattr(item, 'id', '')
    text = getattr(item, 'text', '')
    if item_id == 'msg_answer':
        print('=== AGENT RESPONSE ===')
        print(text)
        print('\n--- Context Carry-Over Checks ---')
        lower = text.lower()
        checks = {
            'References region data': 'region' in lower,
            'References premium data': 'premium' in lower,
            'References claims data': 'claim' in lower,
            'Addresses underpricing': any(w in lower for w in ['underpric', 'priced', 'pricing']),
        }
        for check, passed in checks.items():
            print(f"  [{'PASS' if passed else 'CHECK'}] {check}")
    elif item_id == 'msg_metadata' and text.strip():
        try:
            meta = json.loads(text)
            print(f"  Intent: {meta.get('intent')}  Domain: {meta.get('domain')}")
            print(f"  Nodes:  {meta.get('nodes_executed')}")
        except json.JSONDecodeError:
            pass


### Post-Conversation Memory State


In [ ]:
time.sleep(3)
inspect_memory_tables(CONV_THREAD_ID, CONV_USER_ID, label='AFTER 3-TURN CONVERSATION')

print('--- Checkpoint message count ---')
cp = _load_checkpoint(CONV_THREAD_ID)
if cp and cp.get('messages'):
    print(f"  {len(cp['messages'])} messages stored in latest checkpoint")
    for i, msg in enumerate(cp['messages']):
        print(f"    [{i}] {msg.get('role', '?')}: {msg.get('content', '')[:100]}...")
else:
    print('  No checkpoint found')
